# Module 5: Deploy (~5 min)

Package the Dussault as a serverless endpoint via **Bedrock AgentCore**.

Same pattern as NFL Next Gen Stats: event-driven Lambda that fires on request,
scales to zero between queries, and handles JSON payloads. One inference per
invocation — stateless, horizontally scalable, pay-per-use.

In [ ]:
!pip install -q strands-agents strands-agents-tools bedrock-agentcore

## The BedrockAgentCoreApp Pattern

AgentCore gives you a managed runtime for strands agents. The contract:

- `BedrockAgentCoreApp()` — creates the application shell
- `@app.entrypoint` — decorates the function that receives JSON payloads
- Singleton agent — created once, reused across invocations (warm container)
- `app.run()` — starts the runtime (local dev server or Lambda handler)

The payload arrives as `{"prompt": "your question"}`. The entrypoint
extracts the prompt, runs the agent, returns the string response.

In [ ]:
# main.py structure — the full deploy artifact

import sys
sys.path.insert(0, "../shared")
sys.path.insert(0, "../01-agent-loop-tools")

from strands import Agent
from model_provider import get_model
from strands.agent.conversation_manager import SlidingWindowConversationManager
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from dussault_tools import lookup_player, get_roster_by_position, get_game_result, get_season_stats, get_coaching_staff

app = BedrockAgentCoreApp()

SYSTEM_PROMPT = """You are Dussault, a 2004 New England Patriots Dussault API.
You receive questions about the 2004 season and return evidence-based answers.

When answering:
- Always look up the data before making claims.
- Be specific: cite game weeks, scores, stat lines.
- Connect facts to narrative.
- If the data isn't in your tools, say so clearly."""

_agent = None


def get_agent():
    """Singleton — one agent per container, reused across invocations."""
    global _agent
    if _agent is None:
        _agent = Agent(
            model=get_model(),
            tools=[lookup_player, get_roster_by_position, get_game_result, get_season_stats, get_coaching_staff],
            system_prompt=SYSTEM_PROMPT,
            conversation_manager=SlidingWindowConversationManager(window_size=20),
            callback_handler=None,
        )
    return _agent


@app.entrypoint
def invoke(payload, context):
    """Receives {\"prompt\": \"question\"} — returns string answer."""
    prompt = payload.get("prompt")
    if not prompt:
        raise ValueError("Missing required field: prompt")
    agent = get_agent()
    response = agent(prompt)
    return str(response).strip()


print("\u2705 main.py structure defined — ready to deploy")

## Test Locally Before Deploy

Always verify your agent works end-to-end before deploying. The `invoke`
function accepts the same payload shape the serverless runtime will receive.

In [ ]:
# Local invocation — same payload shape as production
result = invoke({"prompt": "Who was Super Bowl XXXIX MVP?"}, None)
print(result)

## Deploy

```
agentcore deploy
```

Architecture:

```
Client request (JSON)
       │
       ▼
AgentCore Runtime (managed Lambda)
       │
       ▼
@app.entrypoint invoke(payload, context)
       │
       ▼
get_agent() → singleton Agent + get_model()
       │
       ▼
Tool calls (lookup_player, get_game_result, ...)
       │
       ▼
String response → Client
```

The deployed endpoint behaves identically to the local test above.
Scales to zero when idle. Warm-starts reuse the singleton agent.

In [ ]:
# Deploy command (uncomment to run):
# !agentcore deploy

# After deployment, invoke remotely:
# !agentcore invoke --payload '{"prompt": "What was the Patriots record in 2004?"}'

## What's Next

**Module 6** adds **multi-agent orchestration** — a lean coordinator that
delegates podcast research to a specialist agent. Same pattern as NGS
separating play-tracking inference from broadcast graphics inference:
each specialist owns its domain, the orchestrator routes.